# Kitchen Sound Classifier
### Multi-class detection: Frying | Whistle | Ambient

**Dataset structure expected:**
```
Train/
  frying/        ← 100 audio files
  whistle/       ← 100 audio files
  ambient/       ← 100 audio files
Test/
  frying/
  whistle/
  ambient/
```

**Output:** `kitchen_sound_classifier.onnx` — a single ONNX model with a 3-class softmax output.
Class indices: `0 = ambient`, `1 = frying`, `2 = whistle`

In [ ]:
!pip install tf2onnx onnx librosa scikit-learn -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import glob
from typing import List, Tuple
from collections import defaultdict

import tf2onnx
import onnx

import numpy as np
import librosa
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras import layers, models

print("TF:", tf.__version__)

## Configuration

In [ ]:
# ===================== CONFIG =====================
SAMPLE_RATE       = 48000
TARGET_DURATION   = 3.0
FRAME_LENGTH_SEC  = 0.025
FRAME_HOP_SEC     = 0.010
N_MELS            = 128
TARGET_FRAMES     = int(TARGET_DURATION / FRAME_HOP_SEC)   # 300

# Fixed dB normalisation range – must match C++ inference exactly
DB_MIN = -80.0
DB_MAX =   0.0

# 3-class problem
CLASS_NAMES = ["ambient", "frying", "whistle"]
LABEL_MAP   = {name: idx for idx, name in enumerate(CLASS_NAMES)}
NUM_CLASSES = len(CLASS_NAMES)

TRAIN_DATA_DIR    = "/content/drive/MyDrive/CNet_proj_dataset/Train"
TEST_DATA_DIR     = "/content/drive/MyDrive/CNet_proj_dataset/Test"
GOOGLE_DRIVE_PATH = "/content/drive/MyDrive/"
MODEL_BASENAME    = "kitchen_sound_classifier"

print(f"Classes : {CLASS_NAMES}")
print(f"Target frames : {TARGET_FRAMES}")

## Augmentation

In [ ]:
# ===================== AUGMENTATION =====================
def augment_audio(y: np.ndarray, sr: int) -> np.ndarray:
    """Time-domain augmentation – applied when augment=True."""
    # Random gain
    y = y * np.random.uniform(0.6, 1.4)
    # Additive noise
    y = y + np.random.uniform(0.001, 0.015) * np.random.randn(len(y))
    # Time shift ±200 ms
    shift = np.random.randint(-int(0.2 * sr), int(0.2 * sr))
    y = np.roll(y, shift)
    # Pitch shift ±2 semitones
    steps = np.random.uniform(-2.0, 2.0)
    y = librosa.effects.pitch_shift(y, sr=sr, n_steps=steps)
    return y


def spec_augment(S: np.ndarray) -> np.ndarray:
    """Two time-masks + two frequency-masks (SpecAugment)."""
    S = S.copy()
    for _ in range(2):
        t_w = np.random.randint(5, max(6, S.shape[1] // 5))
        t0  = np.random.randint(0, S.shape[1] - t_w)
        S[:, t0:t0 + t_w] = 0.0

        f_w = np.random.randint(5, max(6, S.shape[0] // 5))
        f0  = np.random.randint(0, S.shape[0] - f_w)
        S[f0:f0 + f_w, :] = 0.0
    return S

## Feature Extraction

In [ ]:
# ===================== FEATURE EXTRACTION =====================
def extract_spectrogram_tensor(
    audio_path: str,
    augment: bool = False,
) -> np.ndarray:
    """
    Load audio → compute log-mel spectrogram → fixed-length pad/crop
    → fixed-range normalise → shape (N_MELS, TARGET_FRAMES, 1).

    The DB_MIN / DB_MAX normalisation is IDENTICAL to the C++ runtime
    so there is no train/inference mismatch.
    """
    y, sr = librosa.load(audio_path, sr=SAMPLE_RATE, duration=TARGET_DURATION)

    if augment:
        y = augment_audio(y, sr)

    n_fft      = int(FRAME_LENGTH_SEC * sr)   # 1200
    hop_length = int(FRAME_HOP_SEC * sr)       # 480

    S = librosa.feature.melspectrogram(
        y=y, sr=sr,
        n_mels=N_MELS,
        n_fft=n_fft,
        hop_length=hop_length,
        center=False,
    )
    S_dB = librosa.power_to_db(S, ref=np.max)

    if augment:
        S_dB = spec_augment(S_dB)

    # ── Fix length to TARGET_FRAMES ──────────────────────────────────────
    n_frames = S_dB.shape[1]
    if n_frames > TARGET_FRAMES:
        start = np.random.randint(0, n_frames - TARGET_FRAMES) if augment else 0
        S_dB  = S_dB[:, start:start + TARGET_FRAMES]
    elif n_frames < TARGET_FRAMES:
        S_dB  = np.pad(
            S_dB,
            ((0, 0), (0, TARGET_FRAMES - n_frames)),
            mode="constant",
            constant_values=DB_MIN,
        )

    # ── Fixed-range normalisation (matches C++ build_features) ───────────
    S_norm = (S_dB - DB_MIN) / (DB_MAX - DB_MIN)
    S_norm = np.clip(S_norm, 0.0, 1.0)

    return np.expand_dims(S_norm, axis=-1).astype("float32")

## Dataset Loading

In [ ]:
# ===================== DATASET =====================
def _collect_audio_files(folder: str) -> List[str]:
    files = []
    for p in ["*.wav", "*.mp3", "*.flac", "*.ogg", "*.m4a"]:
        files.extend(glob.glob(os.path.join(folder, p)))
    return sorted(files)


def _collect_dataset_entries(base_dir: str) -> List[Tuple[str, int]]:
    entries = []
    for class_name, label in LABEL_MAP.items():
        label_dir = os.path.join(base_dir, class_name)
        if not os.path.isdir(label_dir):
            print(f"⚠  Folder not found, skipping: {label_dir}")
            continue
        files = _collect_audio_files(label_dir)
        print(f"  {class_name:10s} : {len(files)} files")
        for audio_path in files:
            entries.append((audio_path, label))
    return entries


def load_dataset(base_dir: str, augment: bool = False):
    """Load all audio from base_dir and return (X, y) arrays."""
    X, y = [], []
    for audio_path, label in _collect_dataset_entries(base_dir):
        try:
            X.append(extract_spectrogram_tensor(audio_path, augment=augment))
            y.append(label)
        except Exception as exc:
            print(f"Skipping {audio_path}: {exc}")
    if not X:
        raise RuntimeError(f"No audio files found in '{base_dir}'.")
    return np.stack(X), np.array(y, dtype="int32")

## Model Architecture
MobileNet-style depthwise separable CNN — lightweight and generalises well on small datasets (300 samples).

In [ ]:
# ===================== MODEL =====================
def _dw_block(x, filters: int, strides=(2, 2)):
    """Depthwise-separable block with BatchNorm + ReLU."""
    x = layers.DepthwiseConv2D(3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Conv2D(filters, 1, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    if strides != (1, 1):
        x = layers.MaxPooling2D(strides)(x)
    return x


def build_model(input_shape):
    """
    Input  : (N_MELS, TARGET_FRAMES, 1) = (128, 300, 1)
    Output : softmax over NUM_CLASSES = 3  [ambient, frying, whistle]
    """
    inp = layers.Input(shape=input_shape, name="melspectrogram_input")

    # Stem
    x = layers.Conv2D(16, (3, 3), padding="same", use_bias=False)(inp)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D((2, 2))(x)

    # Depthwise separable blocks
    x = _dw_block(x, 32)
    x = _dw_block(x, 64)
    x = _dw_block(x, 128)
    x = _dw_block(x, 128, strides=(1, 1))  # extra capacity without halving

    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(
        64, activation="relu",
        kernel_regularizer=tf.keras.regularizers.l2(1e-3),
    )(x)
    x = layers.Dropout(0.4)(x)

    # 3-class softmax — output shape (batch, 3)
    out = layers.Dense(
        NUM_CLASSES, activation="softmax", name="class_output"
    )(x)

    model = models.Model(inp, out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=3e-4),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


# Quick architecture preview
dummy = build_model((N_MELS, TARGET_FRAMES, 1))
dummy.summary()

## Training

In [ ]:
# ===================== TRAIN =====================
def train_model(
    data_dir:   str,
    epochs:     int   = 60,
    batch_size: int   = 16,
    val_split:  float = 0.2,
    aug_copies: int   = 4,
):
    """
    Train a 3-class kitchen sound classifier.

    aug_copies  : how many augmented versions to generate per original file.
                  Total training samples ≈ n_originals × (1 + aug_copies).
    """
    print("\n=== Collecting dataset entries ===")
    entries = _collect_dataset_entries(data_dir)
    if not entries:
        raise RuntimeError(f"No audio files found in '{data_dir}'.")

    audio_paths = [p for p, _ in entries]
    labels      = np.array([l for _, l in entries], dtype="int32")

    # ── Split ORIGINAL files before augmentation (avoids data leakage) ──
    train_paths, val_paths, y_train_base, y_val = train_test_split(
        audio_paths, labels,
        test_size=val_split, stratify=labels, random_state=42,
    )
    print(f"Original  train: {len(train_paths)}  |  val: {len(val_paths)}")

    # ── Build augmented training set ─────────────────────────────────────
    train_specs, train_labels = [], []
    for path, label in zip(train_paths, y_train_base):
        train_specs.append(extract_spectrogram_tensor(path, augment=False))
        train_labels.append(label)
        for _ in range(aug_copies):
            train_specs.append(extract_spectrogram_tensor(path, augment=True))
            train_labels.append(label)

    X_train = np.stack(train_specs)
    y_train  = np.array(train_labels, dtype="int32")

    X_val = np.stack([
        extract_spectrogram_tensor(p, augment=False) for p in val_paths
    ])
    y_val = np.array(y_val, dtype="int32")

    print(f"After augmentation – train: {len(X_train)}  val: {len(X_val)}")
    print(f"Train class counts : {np.bincount(y_train)}  (ambient / frying / whistle)")
    print(f"Val   class counts : {np.bincount(y_val)}")

    # ── Balanced class weights ────────────────────────────────────────────
    weights       = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
    class_weights = dict(zip(np.unique(y_train), weights))
    print(f"Class weights : {class_weights}")

    # ── Model ─────────────────────────────────────────────────────────────
    model = build_model(X_train.shape[1:])

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_accuracy", patience=12,
            restore_best_weights=True, verbose=1,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5,
            patience=4, min_lr=1e-6, verbose=1,
        ),
        tf.keras.callbacks.ModelCheckpoint(
            filepath="/content/best_kitchen_sound.keras",
            monitor="val_accuracy", save_best_only=True,
            save_weights_only=False, verbose=1,
        ),
    ]

    model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        class_weight=class_weights,
        callbacks=callbacks,
    )

    # Reload best checkpoint
    model = tf.keras.models.load_model("/content/best_kitchen_sound.keras")

    # ── Validation report ─────────────────────────────────────────────────
    val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)
    print(f"\nBest val accuracy : {val_acc:.4f}  |  val loss : {val_loss:.4f}")

    preds = np.argmax(model.predict(X_val), axis=1)
    print("\nClassification report (validation):")
    print(classification_report(y_val, preds, target_names=CLASS_NAMES))
    print("Confusion matrix (rows=true, cols=pred):")
    print(confusion_matrix(y_val, preds))

    return model

## ONNX Export

In [ ]:
# ===================== ONNX EXPORT =====================
def export_onnx(
    model,
    drive_path: str = GOOGLE_DRIVE_PATH,
    basename:   str = MODEL_BASENAME,
):
    """
    Export Keras model → ONNX opset 13.
    Output shape: (batch, 3)  → class probabilities [ambient, frying, whistle]
    """
    if not os.path.isdir(drive_path):
        print(f"⚠  Drive path not found – saving to /content/")
        drive_path = "/content/"

    onnx_path = os.path.join(drive_path, f"{basename}.onnx")

    input_shape = model.inputs[0].shape
    input_sig   = [
        tf.TensorSpec(
            shape=(None,) + tuple(input_shape[1:]),
            dtype=tf.float32,
            name="melspectrogram_input",
        )
    ]

    try:
        tf2onnx.convert.from_keras(
            model,
            input_signature=input_sig,
            output_path=onnx_path,
            opset=13,
        )
        onnx_model = onnx.load(onnx_path)
        onnx.checker.check_model(onnx_model)
        print(f"✅ ONNX saved and validated : {onnx_path}")
    except Exception as e:
        print(f"❌ ONNX export failed : {e}")
        raise

    return onnx_path

## Sanity Check (optional – run before full training)

In [ ]:
# ===================== SANITY CHECK =====================
def sanity_check(
    data_dir: str,
    samples_per_class: int = 15,
    overfit_epochs:    int = 80,
):
    """
    Overfit a tiny subset of the data.
    Train accuracy should reach >90 %; if not, check your data labels.
    """
    entries = _collect_dataset_entries(data_dir)
    per_class = defaultdict(int)
    selected, selected_labels = [], []

    for path, label in entries:
        if per_class[label] < samples_per_class:
            selected.append(path)
            selected_labels.append(label)
            per_class[label] += 1

    X = np.stack([extract_spectrogram_tensor(p) for p in selected])
    y = np.array(selected_labels, dtype="int32")

    idx = np.random.permutation(len(y))
    X, y = X[idx], y[idx]

    model = build_model(X.shape[1:])
    model.fit(X, y, epochs=overfit_epochs, batch_size=8, verbose=2)

    loss, acc = model.evaluate(X, y, verbose=0)
    print(f"\nSanity result : train acc={acc:.4f}  loss={loss:.4f}")
    if acc < 0.85:
        print("⚠  Cannot overfit — check for mislabelled data!")
    else:
        print("✅ Can overfit — data is fine; tune regularisation for generalisation.")

## Run Everything

In [ ]:
# ── Step 1 (optional) : sanity check ─────────────────────────────────────
# sanity_check(TRAIN_DATA_DIR)

# ── Step 2 : train ───────────────────────────────────────────────────────
model = train_model(
    TRAIN_DATA_DIR,
    epochs=60,
    batch_size=16,
    val_split=0.2,
    aug_copies=4,
)

In [ ]:
# ── Step 3 : evaluate on held-out test set ───────────────────────────────
if os.path.isdir(TEST_DATA_DIR):
    print("\n=== Test set evaluation ===")
    X_test, y_test = load_dataset(TEST_DATA_DIR, augment=False)
    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
    print(f"Test accuracy : {test_acc:.4f}  |  test loss : {test_loss:.4f}")
    preds = np.argmax(model.predict(X_test), axis=1)
    print(classification_report(y_test, preds, target_names=CLASS_NAMES))
    print("Confusion matrix:")
    print(confusion_matrix(y_test, preds))
else:
    print(f"Test folder not found at {TEST_DATA_DIR} — skipping test evaluation.")

In [ ]:
# ── Step 4 : export ONNX ─────────────────────────────────────────────────
onnx_path = export_onnx(model)
print(f"\nONNX model path : {onnx_path}")
print("\nCopy this file to your Raspberry Pi / embedded device and point")
print("predict_rt2.cpp at it with:  --model kitchen_sound_classifier.onnx")

## Inference test (verify ONNX output matches Keras)

In [ ]:
import onnxruntime as ort

def test_onnx_vs_keras(keras_model, onnx_path: str, data_dir: str, n_samples: int = 10):
    """
    Load a few samples, run through both Keras and ONNX, compare probabilities.
    Max absolute difference should be < 1e-4 for a correctly exported model.
    """
    sess = ort.InferenceSession(onnx_path)
    input_name = sess.get_inputs()[0].name

    entries = _collect_dataset_entries(data_dir)[:n_samples]
    max_diff = 0.0

    for audio_path, true_label in entries:
        spec  = extract_spectrogram_tensor(audio_path, augment=False)
        batch = np.expand_dims(spec, axis=0)

        keras_probs = keras_model.predict(batch, verbose=0)[0]
        onnx_probs  = sess.run(None, {input_name: batch})[0][0]

        diff = np.max(np.abs(keras_probs - onnx_probs))
        max_diff = max(max_diff, diff)

        pred_k = CLASS_NAMES[np.argmax(keras_probs)]
        pred_o = CLASS_NAMES[np.argmax(onnx_probs)]
        true_n = CLASS_NAMES[true_label]

        print(f"  True: {true_n:8s}  Keras: {pred_k:8s}  ONNX: {pred_o:8s}  max_diff={diff:.2e}")

    print(f"\nOverall max probability difference : {max_diff:.2e}")
    if max_diff < 1e-4:
        print("✅ ONNX export looks correct.")
    else:
        print("⚠  Large difference detected — check the export.")


test_onnx_vs_keras(model, onnx_path, TRAIN_DATA_DIR)